# Indicator substitution: source-to-results replication

Use **Python 3.12**, an Internet connection, and **Run All**. This notebook downloads the analysis code and public source data, reconstructs the samples, runs the original country-holdout prediction analysis, and exports **four main tables, eight supplementary tables and three figures**. The full model run previously took approximately **22 minutes**. Each fitting job reports elapsed time and estimated time remaining.

The paper reports the results recorded for the submission package dated **5 October 2026**. Re-running the analysis with subsequently updated source data may yield different estimates. This notebook records the retrieval date, source versions, sample sizes and differences from the paper's archived results.

The default `DATA_VERSION = "current"` downloads current World Bank extracts. Choose `"paper"` to use the attributed July/September 2026 World Bank snapshots. NIQ editions, the HLO replication commit and the published reconstruction workbook remain explicitly versioned. The estimation rules, country weighting, folds and tuning grids are the paper's original implementation.

In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
import subprocess
import sys

DATA_VERSION = "current"
OUTPUT = Path("replication_output").resolve()
CODE_COMMIT = "4249fceee31fd782888d42f38e61016ad8c2650a"
CODE_DIRECTORY = Path("replication_code_" + CODE_COMMIT[:8]).resolve()
URL = "https://codeload.github.com/yiorgos1973/indicator-substitution-replication/zip/" + CODE_COMMIT

with urlopen(URL, timeout=180) as response:
    archive_bytes = response.read()
with ZipFile(BytesIO(archive_bytes)) as archive:
    archive.extractall(CODE_DIRECTORY)
CODE_ROOT = CODE_DIRECTORY / ("indicator-substitution-replication-" + CODE_COMMIT)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(CODE_ROOT / "requirements_notebook.txt")], check=True)
sys.path.insert(0, str(CODE_ROOT))

from replication_pipeline import Replication
run = Replication(OUTPUT, data_version=DATA_VERSION)
print("Output:", OUTPUT)
print("Analysis code commit:", CODE_COMMIT)

## Download source data

NIQ v1.3.5 and v1.3.3 are downloaded from View on IQ; HLO is downloaded from the Measuring Human Capital replication repository; the earlier-study workbook is downloaded from the publisher. Current World Bank indicators are downloaded from its API. Paper mode selects the attributed archived World Bank ZIPs supplied with the analysis code. Source hashes and retrieval details are saved in `SOURCE_DOWNLOADS.csv` and `DATA_VERSION.json`.

In [ ]:
run.download()

## Reconstruct the indicators and historical comparisons

This step applies the original source reconciliation, aggregation and inclusion rules, then constructs the fixed-sample agreement comparisons, prior correlation reconstructions and period-specific observations.

In [ ]:
run.reconstruct()

## Construct the contextual predictors and modelling samples

Historical World Bank indicators are aggregated using the paper's coverage, timing and weighting rules. The resulting matrices define the primary analysis, historical/context sensitivities and country-deletion jobs.

In [ ]:
run.build_context()

## Reproduce the shared-outcome comparisons

The GDP analysis compares correlations on the common country sample and regenerates the sensitivity estimates, Williams test, Zou interval and paired-country bootstrap intervals. The bootstrap uses 99,999 replications with the original seed.

In [ ]:
run.gdp_results()

## Fit the held-out prediction models

All periods of the held-out country remain outside training. Preprocessing and penalty selection use the training countries, with equal-country weighting. Ridge, OLS and Lasso use the original implementation. Fitting produces predictions, fold assignments, tuning results and model diagnostics.

In [ ]:
run.fit()

## Export tables, figures and comparisons

The empirical displays are regenerated from the reconstructed samples and fitted predictions. Table 1 and the workflow figure describe the fixed methodological definitions. `comparisons/` records differences from archived numerical ledgers; revised provider values are reported as differences rather than treated as unsuccessful reproduction. `VALIDATION.json` records completed jobs, sample sizes and output checks.

In [ ]:
run.report()

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image, SVG

display(pd.read_csv(OUTPUT / "SOURCE_DOWNLOADS.csv"))
display(pd.read_csv(OUTPUT / "VALIDATION.csv"))
for number in range(1, 5):
    print(f"Table {number}")
    display(pd.read_csv(OUTPUT / "tables" / f"Table_{number}.csv"))
display(SVG(filename=str(OUTPUT / "figures" / "METHODOLOGY_Workflow.svg")))
display(Image(filename=str(OUTPUT / "figures" / "INTELLIGENCE_Figure_1.png"), width=900))
display(Image(filename=str(OUTPUT / "figures" / "INTELLIGENCE_Figure_2.png"), width=900))
print(json.dumps(json.loads((OUTPUT / "DATA_VERSION.json").read_text()), indent=2))
print("Supplementary tables: Table_S1.csv through Table_S8.csv in", OUTPUT / "tables")

## Source attribution

NIQ: Becker, *The NIQ-dataset*, v1.3.5 (2023) and v1.3.3 (2019). HLO: Angrist, Djankov, Goldberg and Patrinos (2021), doi:10.1038/s41586-021-03323-7, and the pinned Measuring Human Capital replication files. Earlier-study reconstruction: Warne (2023), doi:10.1007/s40806-022-00351-y, publisher supplement; Henss (2024), doi:10.32388/6IBSKW, preprint v1. World Bank: *World Development Indicators*; the archived extract attribution, dates and CC BY 4.0/World Bank Dataset Terms are recorded in `data/wdi_frozen/ATTRIBUTION.csv` within the downloaded code. Study-specific selections and aggregations are adaptations.

All outputs remain in the local run directory. The two empirical plots retain their historical `INTELLIGENCE_Figure_1/2` filenames and correspond to Figures 2/3 in the Methodology manuscript.